# Time Series Analisys on Geoespatial Data with Python


Author: João Otavio Nascimento Firigato

email: joaootavionf007@gmail.com

LinkedIn: https://www.linkedin.com/in/jo%C3%A3o-otavio-firigato-4876b3aa/


## First instructions:

✅ Access the link to join our private WhatsApp community for students: https://chat.whatsapp.com/EPn27ZgR07lF3e1vnj8FiI


❗ It is important to access the Whatsapp Group to get the Colab Notebooks, as the PDF files are protected from text copying.




# Chapter 16 - Harmonic Time Series Clustering

In this example we will cluster harmonic time series obtained from Google Earth Engine. We start by installing and importing the necessary libraries:


In [ ]:
!pip install rasterio
!pip install tslearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.4/374.4 kB 5.3 MB/s eta 0:00:00


In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project='my-project-1527255156007')

In [ ]:
import folium
from folium import plugins
from IPython.display import Image
import geopandas as gpd
import json
import math
import pandas as pd
from tslearn.clustering import TimeSeriesKMeans
from tslearn.utils import to_time_series_dataset

Let's select our area and generate 60 random points in that area:

In [ ]:
AOI =  ee.Geometry.Polygon(
        [[[-56.37397887990624, -12.737207954893526],
          [-56.37397887990624, -13.300834158918619],
          [-55.37113311574608, -13.300834158918619],
          [-55.37113311574608, -12.737207954893526]]])
points = ee.FeatureCollection.randomPoints(AOI,60)

Let's create a monthly image collection from 2017 to 2019:

In [ ]:
months = ee.List.sequence(1,12)
years = ee.List.sequence(2017, 2019)

In [ ]:
MD_NDVI = ee.ImageCollection('MODIS/MOD09GA_006_NDVI').filterDate('2017-1-1','2019-12-31').filterBounds(AOI).select('NDVI')

Let's visualize our analysis area, with the points created:

In [ ]:
modis_ndvi = MD_NDVI.median().clip(AOI)
mean_ndvi = MD_NDVI.mean().clip(AOI)

In [ ]:
vis_params = {'min': 0, 'max': 1, 'b': ['red', 'yellow','green']}

In [ ]:
basemaps = {
    'Google Maps': folium.TileLayer(
        tiles = 'https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}',
        attr = 'Google',
        name = 'Google Maps',
        overlay = True,
        control = True
    ),
    'Google Satellite': folium.TileLayer(
        tiles = 'https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
        attr = 'Google',
        name = 'Google Satellite',
        overlay = True,
        control = True
    ),
    'Google Terrain': folium.TileLayer(
        tiles = 'https://mt1.google.com/vt/lyrs=p&x={x}&y={y}&z={z}',
        attr = 'Google',
        name = 'Google Terrain',
        overlay = True,
        control = True
    ),
    'Google Satellite Hybrid': folium.TileLayer(
        tiles = 'https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}',
        attr = 'Google',
        name = 'Google Satellite',
        overlay = True,
        control = True
    ),
    'Esri Satellite': folium.TileLayer(
        tiles = 'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr = 'Esri',
        name = 'Esri Satellite',
        overlay = True,
        control = True
    )
}

In [ ]:
def add_ee_layer(self, ee_object, vis_params, name):

    try:
        # display ee.Image()
        if isinstance(ee_object, ee.image.Image):
            map_id_dict = ee.Image(ee_object).getMapId(vis_params)
            folium.raster_layers.TileLayer(
            tiles = map_id_dict['tile_fetcher'].url_format,
            attr = 'Google Earth Engine',
            name = name,
            overlay = True,
            control = True
            ).add_to(self)
        # display ee.ImageCollection()
        elif isinstance(ee_object, ee.imagecollection.ImageCollection):
            ee_object_new = ee_object.mosaic()
            map_id_dict = ee.Image(ee_object_new).getMapId(vis_params)
            folium.raster_layers.TileLayer(
            tiles = map_id_dict['tile_fetcher'].url_format,
            attr = 'Google Earth Engine',
            name = name,
            overlay = True,
            control = True
            ).add_to(self)
        # display ee.Geometry()
        elif isinstance(ee_object, ee.geometry.Geometry):
            folium.GeoJson(
            data = ee_object.getInfo(),
            name = name,
            overlay = True,
            control = True
        ).add_to(self)
        # display ee.FeatureCollection()
        elif isinstance(ee_object, ee.featurecollection.FeatureCollection):
            ee_object_new = ee.Image().paint(ee_object, 0, 2)
            map_id_dict = ee.Image(ee_object_new).getMapId(vis_params)
            folium.raster_layers.TileLayer(
            tiles = map_id_dict['tile_fetcher'].url_format,
            attr = 'Google Earth Engine',
            name = name,
            overlay = True,
            control = True
        ).add_to(self)

    except:
        print("Could not display {}".format(name))

# Add EE drawing method to folium.
folium.Map.add_ee_layer = add_ee_layer

In [ ]:
my_map = folium.Map(location=[-13.0912068,-55.9881647], zoom_start=10)

# Add custom basemaps
basemaps['Google Maps'].add_to(my_map)

# Add the elevation model to the map object.
my_map.add_ee_layer(modis_ndvi, vis_params, 'NDVI')
my_map.add_ee_layer(points.geometry(), {}, 'Points')
my_map.add_child(folium.LayerControl())

# Display the map.
display(my_map)

We can then generate our monthly collection of images:

In [ ]:
def monthly(collection):
  img_coll = ee.ImageCollection([])
  for y in years.getInfo():
    for m in months.getInfo():
      filtered = collection.filter(ee.Filter.calendarRange(y, y, 'year')).filter(ee.Filter.calendarRange(m, m, 'month'))
      filtered = filtered.median()
      img_coll = img_coll.merge(filtered.set('year', y).set('month', m).set('system:time_start', ee.Date.fromYMD(y, m, 1).getInfo()['value']))
  return img_coll

In [ ]:
Monthly_MD = monthly(MD_NDVI)

Now we generate our NDVI harmonic series:

In [ ]:
dependent = 'NDVI'
harmonics = 3
harmonicFrequencies = list(range(1, harmonics+1))

In [ ]:
harmonicFrequencies

[1, 2, 3]

In [ ]:
def getNames (base, lst_freq) :
  name_lst = []
  for i in lst_freq:
    name_lst.append(ee.String(base + str(i)))
  return name_lst

In [ ]:
cosNames = getNames('cos_', harmonicFrequencies);
sinNames = getNames('sin_', harmonicFrequencies);
independents = ee.List(['constant', 't']).cat(cosNames).cat(sinNames);

In [ ]:
def addConstant (image) :
  return image.addBands(ee.Image(1));

In [ ]:
def addTime (image) :
  date = ee.Date(image.get('system:time_start'));
  years = date.difference(ee.Date('1970-01-01'), 'year');
  timeRadians = ee.Image(years.multiply(2 * math.pi));
  return image.addBands(timeRadians.rename('t').float());

In [ ]:
def addHarmonics (image) :
  frequencies = ee.Image.constant(harmonicFrequencies)
  time = ee.Image(image).select('t')
  cosines = time.multiply(frequencies).cos().rename(cosNames)
  sines = time.multiply(frequencies).sin().rename(sinNames)
  return image.addBands(cosines).addBands(sines)

In [ ]:
harmonicMODIS2 = Monthly_MD.map(addConstant).map(addTime).map(addHarmonics);

In [ ]:
harmonicTrend = harmonicMODIS2.select(independents.add(dependent)).reduce(ee.Reducer.linearRegression(independents.length(), 1))

In [ ]:
harmonicTrendCoefficients = harmonicTrend.select('coefficients').arrayProject([0]).arrayFlatten([independents]);

In [ ]:
fittedHarmonic = harmonicMODIS2.map(lambda image : image.addBands(image.select(independents).multiply(harmonicTrendCoefficients).reduce('sum').rename('fitted')));

In [ ]:
print(fittedHarmonic.getInfo())
print(harmonicTrendCoefficients.getInfo())

{'type': 'ImageCollection', 'bands': [], 'features': [{'type': 'Image', 'bands': [{'id': 'NDVI', 'data_type': {'type': 'PixelType', 'precision': 'float', 'min': -1, 'max': 1}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'constant', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 1, 'max': 1}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 't', 'data_type': {'type': 'PixelType', 'precision': 'float'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'cos_1', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'cos_2', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'cos_3', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'sin_1', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPS

In [ ]:
def fitted_harmonic_to_df(fitted_harmonic_coll, fc):

  fitted_values_list = []
  dates_list = []

  for i in range(fitted_harmonic_coll.size().getInfo()):
    image = ee.Image(fitted_harmonic_coll.toList(fitted_harmonic_coll.size()).get(i))
    fitted_value = image.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=fc,  # Assuming 'fc' is your region of interest
        scale=30,
        maxPixels=1e13
    ).get('fitted').getInfo()
    date = image.date().format('YYYY-MM-dd').getInfo()

    fitted_values_list.append(fitted_value)
    dates_list.append(date)


  df = pd.DataFrame({'fitted': fitted_values_list}, index=pd.to_datetime(dates_list))
  return df

For each point, we extract the NDVI harmonic series and add it to our DataFrame:

In [ ]:
df_points = pd.DataFrame([])
for n in range(1,points.size().getInfo() + 1):
  print(n)
  feat = ee.Geometry.Point(points.getInfo()["features"][n-1]['geometry']['coordinates'])
  fitted_df = fitted_harmonic_to_df(fittedHarmonic,feat)
  df_points = pd.concat([df_points, fitted_df], axis=1)

1
2
3


4


5


6


7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49


50
51
52
53
54
55
56
57
58


KeyboardInterrupt: 

With the DataFrame ready, let's apply TimeSeriesKMeans:

In [ ]:
print(df_points)

In [ ]:
X = df_points.values

In [ ]:
formatted_X = to_time_series_dataset(X)
formatted_X.shape

(100, 48, 1)

In [ ]:
km = TimeSeriesKMeans(n_clusters=2, metric="dtw")
labels = km.fit_predict(formatted_X)

In [ ]:
km = TimeSeriesKMeans(n_clusters=3, metric="softdtw")
labels = km.fit_predict(formatted_X)

In [ ]:
labels

array([0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0])

In [ ]:
labels_bis

array([1, 1, 1, 0, 2, 2, 1, 2, 1, 1, 1, 1, 1, 2, 0, 0, 2, 1, 1, 0, 0, 1,
       1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 2, 1, 0, 2, 0, 0, 1, 1, 1, 2, 2, 1,
       0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 2, 1,
       2, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1,
       0, 0, 1, 2, 0, 1, 1, 1, 1, 2, 2, 0])

In [ ]:
newdata = gpd.GeoDataFrame.from_features(points.getInfo()["features"])
newdata['Class'] = labels_bis

In [ ]:
newdata['Class'].values

array([1, 1, 1, 0, 2, 2, 1, 2, 1, 1, 1, 1, 1, 2, 0, 0, 2, 1, 1, 0, 0, 1,
       1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 2, 1, 0, 2, 0, 0, 1, 1, 1, 2, 2, 1,
       0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 2, 1,
       2, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1,
       0, 0, 1, 2, 0, 1, 1, 1, 1, 2, 2, 0])

With the generated classes we present using Folium:

In [ ]:
resultmap = folium.Map(location=[-13.0912068,-55.9881647], zoom_start=10)
basemaps['Google Satellite Hybrid'].add_to(resultmap)

latitudes = list(newdata.geometry.y.values)
longitudes = list(newdata.geometry.x.values)
labels = list(newdata['Class'].values)
for lat, lng, label in zip(latitudes, longitudes, labels):
  if label == 0:
    folium.Marker(
      location = [lat, lng],
      popup = str(label),
      icon = folium.Icon(color='red')
     ).add_to(resultmap)
  elif label == 1:
    folium.Marker(
      location = [lat, lng],
      popup = str(label),
      icon = folium.Icon(color='blue')
     ).add_to(resultmap)
  else:
    folium.Marker(
      location = [lat, lng],
      popup = str(label),
      icon = folium.Icon(color='green')
     ).add_to(resultmap)


vis_params = {'min': 0, 'max': 1}# Add the elevation model to the map object.
resultmap.add_ee_layer(rgb, {}, 'phase (hue), amplitude (sat), ndvi (value)')
resultmap.add_child(folium.LayerControl())

# Display the map.
display(resultmap)

## Thank you! See you in the next Chapter!